In [2]:
!pip install lightgbm


  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)


In [3]:
#setup
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb
import datetime

# --- Setup Paths ---
DATA_PATH = "../data"
MODELS_PATH = "../models"
os.makedirs(MODELS_PATH, exist_ok=True)

TARGET_COL = "isFraud"
RANDOM_STATE = 42

In [4]:
# Load Final Processed Data
try:
    train_df = pd.read_csv(os.path.join(DATA_PATH, "train_final_processed.csv"))
    # Load preprocessing bundle to get feature list if needed, but we'll use train_df.columns
    # preprocessing_bundle = joblib.load(os.path.join(MODELS_PATH, "preprocessing_bundle.pkl"))
except FileNotFoundError:
    print("Error: train_final_processed.csv not found. Please ensure the feature engineering notebook ran successfully.")
    exit()

# Separate features and target
X = train_df.drop(columns=[TARGET_COL, "TransactionID"])
y = train_df[TARGET_COL]

# Drop 'TransactionDT' and other intermediate time columns if they still exist, 
# keeping only the extracted time features.
cols_to_drop_dt = [c for c in X.columns if c.startswith('TransactionDT') or c.startswith('Transaction_day')]
X.drop(columns=cols_to_drop_dt, errors='ignore', inplace=True)

print("Loaded", len(X.columns), "features.")
print("X shape:", X.shape, ", y shape:", y.shape)

Loaded 341 features.
X shape: (590540, 341) , y shape: (590540,)


In [5]:
## 2. LightGBM Model Definition and Training

# --- Calculate Class Weight for Imbalanced Data ---
# Imbalance Ratio: (Number of Non-Fraud) / (Number of Fraud)
pos_weight = (y == 0).sum() / (y == 1).sum()
print("Class Imbalance Ratio (Non-Fraud / Fraud):", round(pos_weight, 2))


# --- LightGBM Hyperparameters (Baseline) ---
lgbm_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 256,
    'max_depth': 12,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'scale_pos_weight': pos_weight,
    'n_jobs': -1,
    'seed': RANDOM_STATE,
    'verbose': -1,
    'device': 'cpu'
}


# --- K-Fold Cross-Validation Training ---
N_SPLITS = 5
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(X.shape[0])
feature_importance_df = pd.DataFrame()
models = []

print("\nStarting", N_SPLITS, "-Fold LightGBM Training...")
start_time = datetime.datetime.now()

for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = lgb.LGBMClassifier(**lgbm_params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    oof_preds[val_index] = model.predict_proba(X_val)[:, 1]
    models.append(model)

    fold_auc = roc_auc_score(y_val, oof_preds[val_index])
    print("Fold", fold + 1, "/", N_SPLITS, "| AUC:", round(fold_auc, 6))

end_time = datetime.datetime.now()
total_time = end_time - start_time
print("\nTraining Complete. Total Time:", total_time)


Class Imbalance Ratio (Non-Fraud / Fraud): 27.58

Starting 5 -Fold LightGBM Training...
Fold 1 / 5 | AUC: 0.962793
Fold 2 / 5 | AUC: 0.964347
Fold 3 / 5 | AUC: 0.963601
Fold 4 / 5 | AUC: 0.964751
Fold 5 / 5 | AUC: 0.965737

Training Complete. Total Time: 0:07:03.134416


In [8]:
## 3. Evaluation and Feature Importance

# --- Final Evaluation ---
overall_auc = roc_auc_score(y, oof_preds)

print("=========================================")
print("OVERALL OOF AUC:", round(overall_auc, 6))
print("=========================================")

# Classification report using threshold = 0.5
# (In fraud detection, this should be tuned later)
oof_labels = (oof_preds > 0.5).astype(int)

print("\nClassification Report (Threshold 0.5):")
print(classification_report(y, oof_labels))


# --- Feature Importance Aggregation (From last trained model) ---
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print("\nTop 20 Features by Importance:")
print(importance.head(20))


# --- Save Model (Optional) ---
joblib.dump(model, os.path.join(MODELS_PATH, "lgbm_baseline_model.pkl"))
print("\nBaseline LightGBM model saved.")


OVERALL OOF AUC: 0.964013

Classification Report (Threshold 0.5):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    569877
           1       0.69      0.75      0.72     20663

    accuracy                           0.98    590540
   macro avg       0.84      0.87      0.85    590540
weighted avg       0.98      0.98      0.98    590540


Top 20 Features by Importance:
                      feature  importance
300                     card1       15243
115            TransactionAmt       13334
15                      card2       12347
239                     addr1       10758
210          Transaction_hour        8440
29                        C13        5817
329  P_emaildomain_target_enc        5723
200                       D15        5235
133                     card5        4881
3         Transaction_weekday        4775
91                      dist1        4446
61                         C1        3847
340       DeviceInfo_freq_enc